In [ ]:
import pandas as pd
import os

DATA_PATH = '../data/raw'

os.listdir(DATA_PATH)

In [ ]:
#  Creating a local variable for the file path that can be accessed throughout the file.
customers = pd.read_csv(f"{DATA_PATH}/olist_customers_dataset.csv")
orders = pd.read_csv(f"{DATA_PATH}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{DATA_PATH}/olist_order_items_dataset.csv")
order_payments = pd.read_csv(f"{DATA_PATH}/olist_order_payments_dataset.csv")
order_review =  pd.read_csv(f"{DATA_PATH}/olist_order_reviews_dataset.csv")
products = pd.read_csv(f"{DATA_PATH}/olist_products_dataset.csv")
sellers = pd.read_csv(f"{DATA_PATH}/olist_sellers_dataset.csv")
category_translation = pd.read_csv(f"{DATA_PATH}/product_category_name_translation.csv")
geolocation = pd.read_csv(f"{DATA_PATH}/olist_geolocation_dataset.csv")

In [ ]:
# Checking the number of rows and columns in the dataset.
datasets ={
    "customers" : customers,
    "orders" : orders,
    "order_items" : order_items,
    "order_payments" : order_payments,
    "order_reviews" : order_review,
    "products" : products,
    "sellers" : sellers,
    "category_translation" : category_translation ,
    "geolocation" : geolocation 
}

for name , df in datasets.items():
    print(f"{name:25} -> {df.shape}")

In [ ]:
# Checking the data types and columns present in datesets
for name , df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()}")
    print(f"{"=" * 60}")
    print("Columns :")
    print(df.columns.tolist())
    print("\nData Types :")
    print(df.dtypes)

In [ ]:
# Checking the missing values in datasets
for name , df in datasets.items():
     print(f"\n{'=' * 60}")
     print(f"{name.upper()}")
     print(f"{"=" * 60}")

     missing_values = df.isnull().sum()
     null_values = missing_values[missing_values>0]

     if null_values.empty:
          print("No null values")
     else:
          print(null_values)

In [ ]:
# Checking duplicate values in datasets 

for name , df in datasets.items():
    dulipcate = df.duplicated().sum()

    print(f"{name:25} -> {dulipcate} duplicate rows") # 25 is spacing with the values

In [ ]:
# Validate whether the primary key column is actually unique and non-null in each dataset

primary_keys = {
    "customers": "customer_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "order_payments": "payment_sequential",
    "order_reviews": "review_id",
    "products": "product_id",
    "sellers": "seller_id",
    "category_translation": "product_category_name",
    "geolocation": "geolocation_zip_code_prefix"
}

for name, pk in primary_keys.items():
    df = datasets[name]

    duplicate_pk = df[pk].duplicated().sum()
    null_pk = df[pk].isnull().sum()

    print(f"{name:25} | PK: {pk}")
    print(f"Duplicate PKs : {duplicate_pk}")
    print(f"Null PKs      : {null_pk}")
    print("-" * 60)

In [ ]:
# Validate any duplicate Order Items ,Payments and Reviews
print(
    "Duplicate (order_id, order_item_id):",
    order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print(
    "Duplicate (order_id, payment_sequential):",
    order_payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum()
)

print(
    "Duplicate (review_id, order_id):",
    order_review.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

In [ ]:
geolocation["geolocation_zip_code_prefix"].nunique()

In [ ]:
geolocation["geolocation_zip_code_prefix"].value_counts().head(10)

In [ ]:
'''
Observation:
Geolocation contains 1,000,163 records but only 19,015 unique
ZIP code prefixes. Therefore, geolocation_zip_code_prefix is
not unique and should not be treated as a standalone primary key.
'''

In [ ]:
# Checking the datatype of date and time
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table, columns in date_columns.items():
    print(f"\n{'=' * 60}")
    print(table.upper())
    print(f"{'=' * 60}")

    for column in columns:
        print(f"\n{column}")
        print(datasets[table][column].head())

In [ ]:
# Converting date string into datetime datatype
order_dates=[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Define the date column in the order items table
order_items_dates =[
    "shipping_limit_date"
]

# Define the date column in the order reviews table
order_reviews_dates=[
    "review_creation_date",
    "review_answer_timestamp"
]

# Converting string cloumn into datetime
orders[order_dates] = orders[order_dates].apply(pd.to_datetime)
order_items[order_items_dates] = order_items[order_items_dates].apply(pd.to_datetime)
order_review[order_reviews_dates] = order_review[order_reviews_dates].apply(pd.to_datetime)

# Check the updated datetype of date 
print(orders[order_dates].dtypes)
print(order_items[order_items_dates].dtypes)
print(order_review[order_reviews_dates].dtypes)

In [ ]:
# Checking if the apporved date before the purchase date

approved_date=(
    orders["order_approved_at"] < orders["order_purchase_timestamp"]
).sum()

print("Approved before purchase", approved_date)

# Checking if the carries date is before the approve date
carrier_date =(
    orders['order_delivered_carrier_date'] < orders['order_approved_at']
 ).sum()
print("Carrier before approved", carrier_date)

# Checking if the customer delivary date is before the carrier date
invalid_delivery =(
    orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']
 ).sum()
print("Delivery before carrier", invalid_delivery)

In [ ]:
# Filter records where carrier date is before approval date
invalid_carrier_data = orders[
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
]

# Display the relevant dates
invalid_carrier_data[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(10)

In [ ]:
# Filter records where customer delivery is before carrier delivery
invalid_delivery_data = orders[
    orders['order_delivered_customer_date'] < orders["order_delivered_carrier_date"]
]

# Display the relevant dates
invalid_delivery_data[
    [
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].head(10)

In [ ]:
# Check status distribution for carrier-before-approval records
print(
    invalid_carrier_data["order_status"]
    .value_counts()
)

# Check status distribution for delivery-before-carrier records
print(
    invalid_delivery_data['order_status']
    .value_counts()
)

In [ ]:
# Recalculate carrier-before-approval records
invalid_carrier_data = orders[
    orders["order_delivered_carrier_date"] < orders["order_approved_at"]
]

# Recalculate delivery-before-carrier records
invalid_delivery_data = orders[
    orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
]

# Display counts
print("Carrier before approval:", len(invalid_carrier_data))
print("Delivery before carrier:", len(invalid_delivery_data))

# Display order status counts
print("\nCarrier before approval - status:")
print(invalid_carrier_data["order_status"].value_counts())

print("\nDelivery before carrier - status:")
print(invalid_delivery_data["order_status"].value_counts())

In [ ]:
# Checking mininum values of products numeric columns
products[
    [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
].min()

In [ ]:
# Counting how many products with zero weight because every product have weight
zero_weight_products =(products['product_weight_g'] == 0).sum()

print(("Product with zero weight :" , zero_weight_products))

In [ ]:
# Checking if any product have negative values
negative_value_product = [
 
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"

]

for column in negative_value_product:
    count = (products[column] < 0).sum()
    print(f"{column:30} -> {count} negative values")

In [ ]:
# Check negative payment values
negative_payments = (order_payments["payment_value"] < 0).sum()

print("Negative payment values:", negative_payments)

In [ ]:
# Check negative values in order item prices
negative_price = (order_items["price"] < 0).sum()

# Check negative values in freight charges
negative_freight = (order_items["freight_value"] < 0).sum()

print("Negative price values:", negative_price)
print("Negative freight values:", negative_freight)

In [ ]:
# Check unique review scores
print(order_review["review_score"].unique())

# Check review scores outside the expected 1-5 range
invalid_scores = (
    (order_review["review_score"] < 1) |
    (order_review["review_score"] > 5)
).sum()

print("Invalid review scores:", invalid_scores)

In [ ]:
# Check minimum payment installments
print(
    "Minimum payment installments:",
    order_payments["payment_installments"].min()
)

# Check invalid installment values
invalid_installments = (
    order_payments["payment_installments"] <= 0
).sum()

print("Invalid payment installments:", invalid_installments)

In [ ]:
# Filter payment records with invalid installments
invalid_installments_data = order_payments[
    order_payments["payment_installments"] <= 0
]

# Display the invalid records
invalid_installments_data

In [ ]:
# Check unique payment types
print(order_payments["payment_type"].unique())

# Count records for each payment type
print(order_payments["payment_type"].value_counts())

In [ ]:
# Check unique order statuses
print(orders["order_status"].unique())

# Count records for each order status
print(orders["order_status"].value_counts())

In [ ]:
# Check negative customer ZIP codes
negative_customer_zip = (
    customers["customer_zip_code_prefix"] < 0
).sum()

# Check negative seller ZIP codes
negative_seller_zip = (
    sellers["seller_zip_code_prefix"] < 0
).sum()

# Check negative geolocation ZIP codes
negative_geo_zip = (
    geolocation["geolocation_zip_code_prefix"] < 0
).sum()

print("Negative customer ZIP codes:", negative_customer_zip)
print("Negative seller ZIP codes:", negative_seller_zip)
print("Negative geolocation ZIP codes:", negative_geo_zip)

In [ ]:
# Check latitude outside valid range
invalid_latitude = (
    (geolocation["geolocation_lat"] < -90) |
    (geolocation["geolocation_lat"] > 90)
).sum()

# Check longitude outside valid range
invalid_longitude = (
    (geolocation["geolocation_lng"] < -180) |
    (geolocation["geolocation_lng"] > 180)
).sum()

print("Invalid latitude values:", invalid_latitude)
print("Invalid longitude values:", invalid_longitude)

In [ ]:
# Check empty strings in important categorical columns

categorical_columns = {
    "customers": ["customer_city", "customer_state"],
    "orders": ["order_status"],
    "order_payments": ["payment_type"],
    "products": ["product_category_name"],
    "sellers": ["seller_city", "seller_state"],
    "category_translation": [
        "product_category_name",
        "product_category_name_english"
    ],
    "geolocation": ["geolocation_city", "geolocation_state"]
}

for table, columns in categorical_columns.items():
    print(f"\n{'=' * 60}")
    print(table.upper())
    print(f"{'=' * 60}")

    for column in columns:
        empty_count = (datasets[table][column] == "").sum()
        print(f"{column:35} → {empty_count} empty values")

In [ ]:
# Calculate price statistics
print(order_items["price"].describe())

In [ ]:
# Exploring the potential outlier threshold using the IQR method.

# Calculate Q1 and Q3
Q1 = order_items["price"].quantile(0.25)
Q3 = order_items["price"].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Calculate upper outlier limit
upper_limit = Q3 + (1.5 * IQR)

# Count potential high-price outliers
outlier_count = (
    order_items["price"] > upper_limit
).sum()

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Upper limit:", upper_limit)
print("Potential price outliers:", outlier_count)